In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/word2vec-nlp-tutorial/testData.tsv.zip
/kaggle/input/competitions/word2vec-nlp-tutorial/sampleSubmission.csv
/kaggle/input/competitions/word2vec-nlp-tutorial/unlabeledTrainData.tsv.zip
/kaggle/input/competitions/word2vec-nlp-tutorial/labeledTrainData.tsv.zip


In [4]:
import os
import sys
import logging
import time
import random

import pandas as pd
import torch
from torch.utils.data import DataLoader, Dataset
import torch.optim as optim
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification
from tqdm.auto import tqdm

# ===================== 固定随机种子，保证复现 =====================
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ===================== Kaggle输入路径（zip包） =====================
PATH_LABELED_TRAIN = "/kaggle/input/competitions/word2vec-nlp-tutorial/labeledTrainData.tsv.zip"
PATH_TEST = "/kaggle/input/competitions/word2vec-nlp-tutorial/testData.tsv.zip"
OUTPUT_CSV = "/kaggle/working/distilbert_native.csv"

# ===================== Dataset 修复类 =====================
class TrainDataset(Dataset):
    def __init__(self, encodings, labels=None):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        if self.labels is not None:
            return len(self.labels)
        return len(next(iter(self.encodings.values())))


class TestDataset(Dataset):
    def __init__(self, encodings, num_samples=0):
        self.encodings = encodings
        self.num_samples = num_samples

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        return item

    def __len__(self):
        return self.num_samples


if __name__ == '__main__':
    logging.basicConfig(format='%(asctime)s: %(levelname)s: %(message)s')
    logging.root.setLevel(logging.INFO)

    # 读取zip内tsv，pandas支持直接读zip压缩文件
    train = pd.read_csv(
        PATH_LABELED_TRAIN,
        header=0,
        delimiter="\t",
        quoting=3
    )
    test = pd.read_csv(
        PATH_TEST,
        header=0,
        delimiter="\t",
        quoting=3
    )

    train_texts = []
    train_labels = []
    for _, row in train.iterrows():
        train_texts.append(row["review"])
        train_labels.append(row["sentiment"])

    test_texts = []
    for _, row in test.iterrows():
        test_texts.append(row["review"])

    # 划分训练集、验证集
    train_texts, val_texts, train_labels, val_labels = train_test_split(
        train_texts, train_labels, test_size=0.2, random_state=SEED
    )

    # tokenizer
    tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')
    train_encodings = tokenizer(train_texts, truncation=True, padding=True)
    val_encodings = tokenizer(val_texts, truncation=True, padding=True)
    test_encodings = tokenizer(test_texts, truncation=True, padding=True)

    train_dataset = TrainDataset(train_encodings, train_labels)
    val_dataset = TrainDataset(val_encodings, val_labels)
    test_dataset = TestDataset(test_encodings, num_samples=len(test_texts))

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"using device: {device}")

    model = DistilBertForSequenceClassification.from_pretrained("distilbert-base-uncased")
    model.to(device)
    model.train()

    train_loader = DataLoader(train_dataset, batch_size=12, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=24, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=24, shuffle=False)

    # 修复：变量名不能叫optim，会覆盖导入模块
    optimizer = optim.AdamW(model.parameters(), lr=5e-5)

    epochs = 3
    for epoch in range(epochs):
        start_time = time.time()
        train_loss_sum = 0.0
        train_acc_sum = 0.0
        train_batch_cnt = 0

        # -------- train --------
        model.train()
        pbar_train = tqdm(total=len(train_loader), desc=f"Epoch {epoch} Train")
        for batch in train_loader:
            train_batch_cnt += 1
            optimizer.zero_grad()

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            loss.backward()
            optimizer.step()

            preds = torch.argmax(outputs.logits.cpu(), dim=1)
            train_acc_sum += accuracy_score(preds, labels.cpu())
            train_loss_sum += loss.item()

            pbar_train.set_postfix({
                "loss": f"{train_loss_sum/train_batch_cnt:.4f}",
                "acc": f"{train_acc_sum/train_batch_cnt:.4f}"
            })
            pbar_train.update(1)
        pbar_train.close()

        # -------- validate 【重要：移到epoch循环，不在train batch内部】 --------
        model.eval()
        val_loss_sum = 0.0
        val_acc_sum = 0.0
        val_batch_cnt = 0

        with torch.no_grad():
            pbar_val = tqdm(total=len(val_loader), desc=f"Epoch {epoch} Val")
            for batch in val_loader:
                val_batch_cnt +=1
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                labels = batch["labels"].to(device)

                outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
                val_loss = outputs.loss
                preds = torch.argmax(outputs.logits.cpu(), dim=1)

                val_acc_sum += accuracy_score(preds, labels.cpu())
                val_loss_sum += val_loss.item()

                pbar_val.set_postfix({
                    "val_loss": f"{val_loss_sum/val_batch_cnt:.4f}",
                    "val_acc": f"{val_acc_sum/val_batch_cnt:.4f}"
                })
                pbar_val.update(1)
            pbar_val.close()

        epoch_cost = time.time() - start_time
        print(f">>> Epoch {epoch} finished | time: {epoch_cost:.2f}s ")
        print(f"    train_loss={train_loss_sum/train_batch_cnt:.4f} train_acc={train_acc_sum/train_batch_cnt:.4f}")
        print(f"    val_loss={val_loss_sum/val_batch_cnt:.4f}   val_acc={val_acc_sum/val_batch_cnt:.4f}\n")


    # -------- test inference --------
    model.eval()
    test_pred = []
    with torch.no_grad():
        pbar_test = tqdm(total=len(test_loader), desc="Prediction")
        for batch in test_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            outputs = model(input_ids, attention_mask=attention_mask)
            batch_pred = torch.argmax(outputs.logits.cpu(), dim=1).numpy().tolist()
            test_pred.extend(batch_pred)
            pbar_test.update(1)
        pbar_test.close()

    # 输出csv到kaggle working
    result_df = pd.DataFrame({
        "id": test["id"],
        "sentiment": test_pred
    })
    result_df.to_csv(OUTPUT_CSV, index=False, quoting=3)
    logging.info(f"Result saved to {OUTPUT_CSV}")
    print(f"✅ 预测文件已输出：{OUTPUT_CSV}")


2026-08-28 01:05:06,805: INFO: HTTP Request: HEAD https://huggingface.co/distilbert-base-uncased/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"
2026-08-28 01:05:06,898: INFO: HTTP Request: GET https://huggingface.co/distilbert-base-uncased/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"
2026-08-28 01:05:06,899: WARNING: Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

2026-08-28 01:05:06,998: INFO: HTTP Request: GET https://huggingface.co/api/models/distilbert-base-uncased/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 307 Temporary Redirect"
2026-08-28 01:05:07,083: INFO: HTTP Request: GET https://huggingface.co/api/models/distilbert/distilbert-base-uncased/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-08-28 01:05:07,170: INFO: HTTP Request: GET https://huggingface.co/api/models/distilbert-base-uncased/tree/main?recursive=true&expand=false "HTTP/1.1 307 Temporary Redirect"
2026-08-28 01:05:07,254: INFO: HTTP Request: GET https://huggingface.co/api/models/distilbert/distilbert-base-uncased/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
2026-08-28 01:05:07,339: INFO: HTTP Request: HEAD https://huggingface.co/distilbert-base-uncased/resolve/main/vocab.txt "HTTP/1.1 200 OK"
2026-08-28 01:05:07,428: INFO: HTTP Request: GET https://huggingface.co/distilbert-base-uncased

vocab.txt: 0.00B [00:00, ?B/s]

2026-08-28 01:05:07,550: INFO: HTTP Request: HEAD https://huggingface.co/distilbert-base-uncased/resolve/main/tokenizer.json "HTTP/1.1 200 OK"
2026-08-28 01:05:07,646: INFO: HTTP Request: GET https://huggingface.co/distilbert-base-uncased/resolve/main/tokenizer.json "HTTP/1.1 200 OK"


...

Epoch 2 Val:   0%|          | 0/209 [00:00<?, ?it/s]

>>> Epoch 2 finished | time: 1101.18s 
    train_loss=0.0739 train_acc=0.9755
    val_loss=0.3996   val_acc=0.8995



Prediction:   0%|          | 0/1042 [00:00<?, ?it/s]

2026-08-28 02:07:03,675: INFO: Result saved to /kaggle/working/distilbert_native.csv


✅ 预测文件已输出：/kaggle/working/distilbert_native.csv


In [3]:
import torch

def check_kaggle_gpu():
    print("===== Kaggle GPU 状态检测 =====")
    has_cuda = torch.cuda.is_available()
    print(f"torch.cuda.is_available(): {has_cuda}")

    if not has_cuda:
        print("⚠️ 未检测到GPU！请在Notebook设置开启 GPU(T4) 加速器")
        return False

    gpu_count = torch.cuda.device_count()
    print(f"GPU数量: {gpu_count}")
    for i in range(gpu_count):
        print(f"\nGPU {i} 名称: {torch.cuda.get_device_name(i)}")
        total_mem = torch.cuda.get_device_properties(i).total_memory / 1024**3
        allocated = torch.cuda.memory_allocated(i) / 1024**3
        reserved = torch.cuda.memory_reserved(i) / 1024**3
        print(f"总显存: {total_mem:.2f} GB")
        print(f"已分配显存: {allocated:.2f} GB")
        print(f"PyTorch预留显存: {reserved:.2f} GB")

    print(f"\n当前使用device: {torch.cuda.current_device()}")
    print("✅ GPU可用")
    return True

# 调用
check_kaggle_gpu()

# 获取device对象直接给模型用
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\ndevice = {device}")


===== Kaggle GPU 状态检测 =====
torch.cuda.is_available(): True
GPU数量: 2

GPU 0 名称: Tesla T4
总显存: 14.56 GB
已分配显存: 0.00 GB
PyTorch预留显存: 0.00 GB

GPU 1 名称: Tesla T4
总显存: 14.56 GB
已分配显存: 0.00 GB
PyTorch预留显存: 0.00 GB

当前使用device: 0
✅ GPU可用

device = cuda


In [7]:
import os
import sys
import logging
import time
import random
import gc

# 屏蔽transformers警告，修复新版本库logging模块导入报错，必须放在transformers导入之前
os.environ["TRANSFORMERS_VERBOSITY"] = "error"

import pandas as pd
import torch
from torch.utils.data import DataLoader, Dataset
import torch.optim as optim
from torch.cuda.amp import autocast, GradScaler
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification
)
from tqdm.auto import tqdm

# ===================== 全局配置 =====================
SEED = 42
MAX_SEQ_LEN = 256
BATCH_SIZE_TRAIN = 12
BATCH_SIZE_VAL_TEST = 24
LR = 5e-5
EPOCHS = 3
GRAD_CLIP = 1.0
EARLY_STOP_PATIENCE = 1

# 固定随机种子
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ===================== Kaggle路径 =====================
PATH_LABELED_TRAIN = "/kaggle/input/competitions/word2vec-nlp-tutorial/labeledTrainData.tsv.zip"
PATH_TEST = "/kaggle/input/competitions/word2vec-nlp-tutorial/testData.tsv.zip"
OUTPUT_CSV = "/kaggle/working/distilbert_native.csv"
BEST_MODEL_PATH = "/kaggle/working/best_distilbert.pt"

# ===================== GPU检测函数 =====================
def check_gpu():
    print("="*40)
    has_cuda = torch.cuda.is_available()
    print(f"CUDA available: {has_cuda}")
    if not has_cuda:
        print("⚠️ WARNING: No GPU detected! Please turn on T4 GPU accelerator!")
        return torch.device("cpu")
    gpu_name = torch.cuda.get_device_name(0)
    total_mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU: {gpu_name}, Total Mem: {total_mem:.2f} GB")
    print("="*40)
    return torch.device("cuda")

# ===================== Dataset =====================
class TrainDataset(Dataset):
    def __init__(self, encodings, labels=None):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        if self.labels is not None:
            return len(self.labels)
        return len(next(iter(self.encodings.values())))


class TestDataset(Dataset):
    def __init__(self, encodings, num_samples=0):
        self.encodings = encodings
        self.num_samples = num_samples

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        return item

    def __len__(self):
        return self.num_samples


if __name__ == '__main__':
    logging.basicConfig(format='%(asctime)s: %(levelname)s: %(message)s')
    logging.root.setLevel(logging.INFO)

    device = check_gpu()

    # 读取数据
    train = pd.read_csv(PATH_LABELED_TRAIN, header=0, delimiter="\t", quoting=3)
    test = pd.read_csv(PATH_TEST, header=0, delimiter="\t", quoting=3)

    train_texts = train["review"].tolist()
    train_labels = train["sentiment"].tolist()
    test_texts = test["review"].tolist()

    train_texts, val_texts, train_labels, val_labels = train_test_split(
        train_texts, train_labels, test_size=0.2, random_state=SEED
    )

    # tokenizer 指定max_length
    tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')
    train_encodings = tokenizer(train_texts, truncation=True, padding="max_length", max_length=MAX_SEQ_LEN)
    val_encodings = tokenizer(val_texts, truncation=True, padding="max_length", max_length=MAX_SEQ_LEN)
    test_encodings = tokenizer(test_texts, truncation=True, padding="max_length", max_length=MAX_SEQ_LEN)

    train_dataset = TrainDataset(train_encodings, train_labels)
    val_dataset = TrainDataset(val_encodings, val_labels)
    test_dataset = TestDataset(test_encodings, num_samples=len(test_texts))

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE_TRAIN, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE_VAL_TEST, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE_VAL_TEST, shuffle=False)

    model = DistilBertForSequenceClassification.from_pretrained("distilbert-base-uncased")
    model.to(device)

    optimizer = optim.AdamW(model.parameters(), lr=LR)
    # 学习率调度：每个epoch线性衰减到0
    lr_scheduler = optim.lr_scheduler.LinearLR(optimizer, start_factor=1.0, end_factor=0.0, total_iters=EPOCHS)
    scaler = GradScaler()  # AMP混合精度

    best_val_loss = float("inf")
    early_stop_counter = 0

    for epoch in range(EPOCHS):
        start_time = time.time()
        train_loss_sum = 0.0
        train_acc_sum = 0.0
        train_batch_cnt = 0

        model.train()
        pbar_train = tqdm(total=len(train_loader), desc=f"Epoch {epoch} Train", leave=False)
        for batch in train_loader:
            train_batch_cnt += 1
            optimizer.zero_grad()

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            # 混合精度 autocast
            with autocast():
                outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
                loss = outputs.loss

            scaler.scale(loss).backward()
            # 梯度裁剪
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)

            scaler.step(optimizer)
            scaler.update()

            preds = torch.argmax(outputs.logits.cpu(), dim=1)
            train_acc_sum += accuracy_score(preds, labels.cpu())
            train_loss_sum += loss.item()

            pbar_train.update(1)
        pbar_train.close()

        # lr step per epoch
        lr_scheduler.step()

        # -------- Validation --------
        model.eval()
        val_loss_sum = 0.0
        val_acc_sum = 0.0
        val_batch_cnt = 0

        with torch.no_grad():
            pbar_val = tqdm(total=len(val_loader), desc=f"Epoch {epoch} Val", leave=False)
            for batch in val_loader:
                val_batch_cnt += 1
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                labels = batch["labels"].to(device)

                with autocast():
                    outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
                    val_loss = outputs.loss

                preds = torch.argmax(outputs.logits.cpu(), dim=1)
                val_acc_sum += accuracy_score(preds, labels.cpu())
                val_loss_sum += val_loss.item()

                pbar_val.update(1)
            pbar_val.close()

        epoch_cost = time.time() - start_time
        avg_train_loss = train_loss_sum / train_batch_cnt
        avg_train_acc = train_acc_sum / train_batch_cnt
        avg_val_loss = val_loss_sum / val_batch_cnt
        avg_val_acc = val_acc_sum / val_batch_cnt

        print(f">>> Epoch {epoch} finished | time: {epoch_cost:.2f}s ")
        print(f"    train_loss={avg_train_loss:.4f} train_acc={avg_train_acc:.4f}")
        print(f"    val_loss={avg_val_loss:.4f}   val_acc={avg_val_acc:.4f}")

        # ========= EarlyStopping & save best model =========
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            early_stop_counter = 0
            # save best state dict
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'best_val_loss': best_val_loss,
            }, BEST_MODEL_PATH)
            print(f"💾 Save best model, val_loss={best_val_loss:.4f}")
        else:
            early_stop_counter += 1
            print(f"⏸ Early stop counter: {early_stop_counter}/{EARLY_STOP_PATIENCE}")
            if early_stop_counter >= EARLY_STOP_PATIENCE:
                print("🛑 Trigger early stopping!")
                break
        print("-"*60)

        # 释放显存
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # ========= 加载最优模型做预测，不用最后epoch权重 =========
    print("\n🔄 Load best checkpoint for test inference ...")
    checkpoint = torch.load(BEST_MODEL_PATH, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()

    test_pred = []
    with torch.no_grad():
        pbar_test = tqdm(total=len(test_loader), desc="Prediction", leave=False)
        for batch in test_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            with autocast():
                outputs = model(input_ids, attention_mask=attention_mask)
            batch_pred = torch.argmax(outputs.logits.cpu(), dim=1).numpy().tolist()
            test_pred.extend(batch_pred)
            pbar_test.update(1)
        pbar_test.close()

    # 输出结果
    result_df = pd.DataFrame({"id": test["id"], "sentiment": test_pred})
    result_df.to_csv(OUTPUT_CSV, index=False, quoting=3)
    logging.info(f"Result saved to {OUTPUT_CSV}")
    print(f"✅ 预测文件已输出：{OUTPUT_CSV}")
    print(f"📊 Prediction distribution: 0:{sum(x==0 for x in test_pred)}, 1:{sum(x==1 for x in test_pred)}")


CUDA available: True
GPU: Tesla T4, Total Mem: 14.56 GB


2026-08-28 02:16:35,553: INFO: HTTP Request: HEAD https://huggingface.co/distilbert-base-uncased/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"
2026-08-28 02:16:35,641: INFO: HTTP Request: GET https://huggingface.co/api/models/distilbert-base-uncased/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 307 Temporary Redirect"
2026-08-28 02:16:35,725: INFO: HTTP Request: GET https://huggingface.co/api/models/distilbert/distilbert-base-uncased/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-08-28 02:16:35,808: INFO: HTTP Request: GET https://huggingface.co/api/models/distilbert-base-uncased/tree/main?recursive=true&expand=false "HTTP/1.1 307 Temporary Redirect"
2026-08-28 02:16:35,896: INFO: HTTP Request: GET https://huggingface.co/api/models/distilbert/distilbert-base-uncased/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
2026-08-28 02:16:55,937: INFO: HTTP Request: HEAD https://huggingface.co/distilbert

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/tmp/ipykernel_58/1097235529.py:129: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()  # AMP混合精度


Epoch 0 Train:   0%|          | 0/1667 [00:00<?, ?it/s]

...

/tmp/ipykernel_58/1097235529.py:187: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/tmp/ipykernel_58/1097235529.py:187: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/tmp/ipykernel_58/1097235529.py:187: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/tmp/ipykernel_58/1097235529.py:187: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/tmp/ipykernel_58/1097235529.py:187: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/tmp/ipykernel_58/1097235529.py:187: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Ple

>>> Epoch 1 finished | time: 164.34s 
    train_loss=0.1859 train_acc=0.9428
    val_loss=0.3290   val_acc=0.9067
⏸ Early stop counter: 1/1
🛑 Trigger early stopping!

🔄 Load best checkpoint for test inference ...


Prediction:   0%|          | 0/1042 [00:00<?, ?it/s]

/tmp/ipykernel_58/1097235529.py:245: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
2026-08-28 02:23:12,931: INFO: Result saved to /kaggle/working/distilbert_native.csv


✅ 预测文件已输出：/kaggle/working/distilbert_native.csv
📊 Prediction distribution: 0:13528, 1:11472


In [10]:
import os
import sys
import logging
import time
import random
import gc
import warnings

# 全局过滤警告，禁止刷屏
warnings.filterwarnings("ignore")
os.environ["TRANSFORMERS_VERBOSITY"] = "error"

import pandas as pd
import torch
from torch.utils.data import DataLoader, Dataset
import torch.optim as optim
from torch.cuda.amp import autocast, GradScaler
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    get_scheduler
)
from tqdm.auto import tqdm

# ===================== 全局配置 =====================
SEED = 42
MAX_SEQ_LEN = 256
BATCH_SIZE_TRAIN = 16
BATCH_SIZE_VAL_TEST = 32
LR = 3e-5
EPOCHS = 4
GRAD_CLIP = 1.0
EARLY_STOP_PATIENCE = 2
WARMUP_RATIO = 0.1
WEIGHT_DECAY = 0.01

# 固定随机种子
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ===================== Kaggle路径 =====================
PATH_LABELED_TRAIN = "/kaggle/input/competitions/word2vec-nlp-tutorial/labeledTrainData.tsv.zip"
PATH_TEST = "/kaggle/input/competitions/word2vec-nlp-tutorial/testData.tsv.zip"
OUTPUT_CSV = "/kaggle/working/distilbert_native.csv"
BEST_MODEL_PATH = "/kaggle/working/best_distilbert.pt"

# ===================== GPU检测函数 =====================
def check_gpu():
    print("="*40)
    has_cuda = torch.cuda.is_available()
    print(f"CUDA available: {has_cuda}")
    if not has_cuda:
        print("⚠️ WARNING: No GPU detected! Please turn on T4 GPU accelerator!")
        return torch.device("cpu")
    gpu_name = torch.cuda.get_device_name(0)
    total_mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU: {gpu_name}, Total Mem: {total_mem:.2f} GB")
    print("="*40)
    return torch.device("cuda")

# ===================== Dataset =====================
class TrainDataset(Dataset):
    def __init__(self, encodings, labels=None):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        if self.labels is not None:
            return len(self.labels)
        return len(next(iter(self.encodings.values())))


class TestDataset(Dataset):
    def __init__(self, encodings, num_samples=0):
        self.encodings = encodings
        self.num_samples = num_samples

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        return item

    def __len__(self):
        return self.num_samples


if __name__ == '__main__':
    logging.basicConfig(format='%(asctime)s: %(levelname)s: %(message)s')
    logging.root.setLevel(logging.INFO)

    device = check_gpu()

    # 读取数据
    train = pd.read_csv(PATH_LABELED_TRAIN, header=0, delimiter="\t", quoting=3)
    test = pd.read_csv(PATH_TEST, header=0, delimiter="\t", quoting=3)

    train_texts = train["review"].tolist()
    train_labels = train["sentiment"].tolist()
    test_texts = test["review"].tolist()

    train_texts, val_texts, train_labels, val_labels = train_test_split(
        train_texts, train_labels, test_size=0.2, random_state=SEED, shuffle=True
    )

    tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')
    train_encodings = tokenizer(train_texts, truncation=True, padding="max_length", max_length=MAX_SEQ_LEN)
    val_encodings = tokenizer(val_texts, truncation=True, padding="max_length", max_length=MAX_SEQ_LEN)
    test_encodings = tokenizer(test_texts, truncation=True, padding="max_length", max_length=MAX_SEQ_LEN)

    train_dataset = TrainDataset(train_encodings, train_labels)
    val_dataset = TrainDataset(val_encodings, val_labels)
    test_dataset = TestDataset(test_encodings, num_samples=len(test_texts))

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE_TRAIN, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE_VAL_TEST, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE_VAL_TEST, shuffle=False)

    model = DistilBertForSequenceClassification.from_pretrained("distilbert-base-uncased")
    model.to(device)

    # BERT分组权重衰减
    no_decay = ["bias", "LayerNorm.weight"]
    optimizer_grouped_parameters = [
        {
            "params": [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)],
            "weight_decay": WEIGHT_DECAY,
        },
        {
            "params": [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)],
            "weight_decay": 0.0,
        },
    ]
    optimizer = optim.AdamW(optimizer_grouped_parameters, lr=LR)

    num_training_steps = EPOCHS * len(train_loader)
    num_warmup_steps = int(num_training_steps * WARMUP_RATIO)
    lr_scheduler = get_scheduler(
        name="cosine",
        optimizer=optimizer,
        num_warmup_steps=num_warmup_steps,
        num_training_steps=num_training_steps
    )

    scaler = GradScaler()
    best_val_loss = float("inf")
    early_stop_counter = 0

    for epoch in range(EPOCHS):
        start_time = time.time()
        train_loss_sum = 0.0
        train_acc_sum = 0.0
        train_batch_cnt = 0

        model.train()
        pbar_train = tqdm(total=len(train_loader), desc=f"Epoch {epoch} Train", leave=False)
        for batch in train_loader:
            train_batch_cnt += 1
            optimizer.zero_grad()

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            # 兼容旧版pytorch，去掉device_type
            with autocast():
                outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
                loss = outputs.loss

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)

            scaler.step(optimizer)
            scaler.update()
            lr_scheduler.step()

            preds = torch.argmax(outputs.logits.cpu(), dim=1)
            train_acc_sum += accuracy_score(preds, labels.cpu())
            train_loss_sum += loss.item()

            pbar_train.update(1)
        pbar_train.close()

        # -------- Validation --------
        model.eval()
        val_loss_sum = 0.0
        val_acc_sum = 0.0
        val_batch_cnt = 0

        with torch.no_grad():
            pbar_val = tqdm(total=len(val_loader), desc=f"Epoch {epoch} Val", leave=False)
            for batch in val_loader:
                val_batch_cnt += 1
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                labels = batch["labels"].to(device)

                with autocast():
                    outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
                    val_loss = outputs.loss

                preds = torch.argmax(outputs.logits.cpu(), dim=1)
                val_acc_sum += accuracy_score(preds, labels.cpu())
                val_loss_sum += val_loss.item()

                pbar_val.update(1)
            pbar_val.close()

        epoch_cost = time.time() - start_time
        avg_train_loss = train_loss_sum / train_batch_cnt
        avg_train_acc = train_acc_sum / train_batch_cnt
        avg_val_loss = val_loss_sum / val_batch_cnt
        avg_val_acc = val_acc_sum / val_batch_cnt

        print(f">>> Epoch {epoch} finished | time: {epoch_cost:.2f}s ")
        print(f"    train_loss={avg_train_loss:.4f} train_acc={avg_train_acc:.4f}")
        print(f"    val_loss={avg_val_loss:.4f}   val_acc={avg_val_acc:.4f}")

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            early_stop_counter = 0
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'best_val_loss': best_val_loss,
            }, BEST_MODEL_PATH)
            print(f"💾 Save best model, val_loss={best_val_loss:.4f}")
        else:
            early_stop_counter += 1
            print(f"⏸ Early stop counter: {early_stop_counter}/{EARLY_STOP_PATIENCE}")
            if early_stop_counter >= EARLY_STOP_PATIENCE:
                print("🛑 Trigger early stopping!")
                break
        print("-"*70)

        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # 预测
    print("\n🔄 Load best checkpoint for test inference ...")
    checkpoint = torch.load(BEST_MODEL_PATH, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()

    test_pred = []
    with torch.no_grad():
        pbar_test = tqdm(total=len(test_loader), desc="Prediction", leave=False)
        for batch in test_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            with autocast():
                outputs = model(input_ids, attention_mask=attention_mask)
            batch_pred = torch.argmax(outputs.logits.cpu(), dim=1).numpy().tolist()
            test_pred.extend(batch_pred)
            pbar_test.update(1)
        pbar_test.close()

    result_df = pd.DataFrame({"id": test["id"], "sentiment": test_pred})
    result_df.to_csv(OUTPUT_CSV, index=False, quoting=3)
    logging.info(f"Result saved to {OUTPUT_CSV}")
    print(f"✅ Output csv: {OUTPUT_CSV}")
    print(f"📊 Prediction distribution: 0:{sum(x==0 for x in test_pred)}, 1:{sum(x==1 for x in test_pred)}")



CUDA available: True
GPU: Tesla T4, Total Mem: 14.56 GB


2026-08-28 02:39:12,867: INFO: HTTP Request: HEAD https://huggingface.co/distilbert-base-uncased/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"
2026-08-28 02:39:12,950: INFO: HTTP Request: GET https://huggingface.co/api/models/distilbert-base-uncased/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 307 Temporary Redirect"
2026-08-28 02:39:13,036: INFO: HTTP Request: GET https://huggingface.co/api/models/distilbert/distilbert-base-uncased/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-08-28 02:39:13,121: INFO: HTTP Request: GET https://huggingface.co/api/models/distilbert-base-uncased/tree/main?recursive=true&expand=false "HTTP/1.1 307 Temporary Redirect"
2026-08-28 02:39:13,208: INFO: HTTP Request: GET https://huggingface.co/api/models/distilbert/distilbert-base-uncased/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
2026-08-28 02:39:33,985: INFO: HTTP Request: HEAD https://huggingface.co/distilbert

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch 0 Train:   0%|          | 0/1250 [00:00<?, ?it/s]

...

Epoch 3 Val:   0%|          | 0/157 [00:00<?, ?it/s]

>>> Epoch 3 finished | time: 152.48s 
    train_loss=0.0466 train_acc=0.9897
    val_loss=0.4130   val_acc=0.9136
⏸ Early stop counter: 2/2
🛑 Trigger early stopping!

🔄 Load best checkpoint for test inference ...


Prediction:   0%|          | 0/782 [00:00<?, ?it/s]

2026-08-28 02:50:40,602: INFO: Result saved to /kaggle/working/distilbert_native.csv


✅ Output csv: /kaggle/working/distilbert_native.csv
📊 Prediction distribution: 0:12188, 1:12812
